# EHR Timeline Transformer — Colab MEM Training

Run the full **Masked Event Modeling (MEM)** pipeline on Colab:

**prepare_data → MEM pretrain → fine-tune → test predictions**

## Workflow
1. Upload raw data to Google Drive once (see setup cells below).
2. Push MEM code to GitHub locally, then re-run the **Setup** cells (`git pull`).
3. `data/` and `outputs/` persist on Drive across runtime restarts.

## Stages
- **MEM pretrain**: uses only official **train** split; monitoring/early stopping uses an internal `pretrain_val` split (`outputs/processed/pretrain_split.json`). Official `val` is **not** used here.
- **Fine-tune**: official train + val for model selection.
- **Predict**: generate `predictions.csv` for submission.

## Requirements
- **GPU runtime** recommended for pretrain and fine-tune.
- Re-run `prepare_data.py` after vocab changes (ensures `[MASK]` in `vocab.json`).
- Ensure the remote repo contains `scripts/train_pretrain.py` before running pretrain cells.


In [ ]:
# --- Edit these paths for your environment ---
REPO_URL = "https://github.com/ShutingXie/modeling_patient_timelines.git"
REPO_DIR = "modeling_patient_timelines"
REPO_BRANCH = "feature/mem-pretrain"

# Persistent locations on Google Drive (upload data once; reused every session)
DRIVE_PROJECT_DIR = "/content/drive/MyDrive"
DRIVE_DATA_DIR = f"{DRIVE_PROJECT_DIR}/data"
DRIVE_OUTPUTS_DIR = f"{DRIVE_PROJECT_DIR}/outputs"

# Optional: zip with raw data for first-time setup. Set to None after data is extracted.
# DATA_ZIP = f"{DRIVE_PROJECT_DIR}/ehr_data.zip"  # or None


In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive", force_remount=True)
os.chdir(DRIVE_PROJECT_DIR)
print(os.getcwd())


In [ ]:
REQUIRED = [
    "patient_splits.csv",
    "target_conditions.csv",
    "test_anchors.csv",
    "train_val",
    "test",
]


In [ ]:
import zipfile
from pathlib import Path

data_dir = Path(DRIVE_DATA_DIR)
missing = [name for name in REQUIRED if not (data_dir / name).exists()]

if missing:
    if "DATA_ZIP" in globals() and DATA_ZIP and Path(DATA_ZIP).exists():
        print(f"First-time setup: extracting {DATA_ZIP}")
        data_dir.parent.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(DATA_ZIP) as zf:
            names = zf.namelist()
            top_levels = {n.split("/")[0] for n in names if n.strip()}
            if top_levels == {"data"}:
                zf.extractall(data_dir.parent)
            else:
                data_dir.mkdir(parents=True, exist_ok=True)
                zf.extractall(data_dir)
        missing = [name for name in REQUIRED if not (data_dir / name).exists()]
    if missing:
        raise FileNotFoundError(
            f"Missing data in {DRIVE_DATA_DIR}: {', '.join(missing)}. "
            "Upload the raw CSV folders to Drive or set DATA_ZIP."
        )
else:
    print(f"Data ready at {DRIVE_DATA_DIR}")


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

repo_dir = Path(REPO_DIR)

if repo_dir.exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull"], check=False)
    print(f"Updated existing repo at {REPO_DIR}")
else:
    subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
    print(f"Cloned repo to {REPO_DIR}")

subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin"], check=True)
subprocess.run(
    ["git", "-C", str(repo_dir), "checkout", REPO_BRANCH],
    check=True,
)
subprocess.run(
    ["git", "-C", str(repo_dir), "pull", "origin", REPO_BRANCH],
    check=True,
)

subprocess.run(
    ["pip", "install", "-q", "-r", str(repo_dir / "requirements.txt")],
    check=True,
)

# Link persistent Drive folders into the repo (no re-upload on code updates).
for link_name, target in [("data", DRIVE_DATA_DIR), ("outputs", DRIVE_OUTPUTS_DIR)]:
    link = repo_dir / link_name
    target_path = Path(target).resolve()
    target_path.mkdir(parents=True, exist_ok=True)
    if link.is_symlink() and link.resolve() == target_path:
        continue
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        backup = repo_dir / f"{link_name}.colab_backup"
        if backup.exists():
            shutil.rmtree(backup)
        link.rename(backup)
        print(f"Backed up existing {link_name}/ to {backup.name}/")
    link.symlink_to(target_path, target_is_directory=True)
    print(f"Linked {link} -> {target_path}")

os.chdir(repo_dir)
print(f"Working directory: {repo_dir}")


In [ ]:
from pathlib import Path

for name in REQUIRED:
    path = Path("data") / name
    assert path.exists(), f"Missing data/{name}"

print("All required data files are present.")


## Data preparation

Build processed artifacts under `outputs/processed/`. Required before MEM pretrain so `vocab.json` includes the `[MASK]` token.


In [ ]:
!python scripts/prepare_data.py --config configs/transformer.yaml


In [ ]:
import json
from pathlib import Path

vocab_path = Path("outputs/processed/vocab.json")
assert vocab_path.exists(), "Run prepare_data first"

vocab = json.loads(vocab_path.read_text())
token_to_id = vocab["token_to_id"]
assert "[MASK]" in token_to_id, "vocab.json is missing [MASK]; re-run prepare_data after pulling MEM code"

processed = sorted(p.name for p in Path("outputs/processed").iterdir())
print(f"[MASK] token id: {token_to_id['[MASK]']}")
print("Processed artifacts:")
for name in processed:
    print(f"  - {name}")


## MEM pretrain

Pretrain uses only official **train** patients, split internally into `pretrain_train` / `pretrain_val` (see `configs/transformer.yaml` → `pretrain:`).

Outputs:
- `outputs/checkpoints/pretrain_best.pt`
- `outputs/metrics/pretrain_metrics.json`
- `outputs/plots/pretrain_curves.png`
- `outputs/processed/pretrain_split.json`

If `pretrain_best.pt` already exists on Drive, you may skip the pretrain cell and go directly to fine-tune.


In [ ]:
import torch
print(
    "CUDA available:",
    torch.cuda.is_available(),
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
)


In [ ]:
import yaml
from pathlib import Path

cfg_path = Path("configs/transformer.yaml")
cfg = yaml.safe_load(cfg_path.read_text())
cfg["wandb"]["run_name"] = "colab_mem_pretrain_v1"
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(f"Set wandb run_name to {cfg['wandb']['run_name']}")


In [ ]:
import wandb
wandb.login()

!python scripts/train_pretrain.py --config configs/transformer.yaml


In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

metrics_path = Path("outputs/metrics/pretrain_metrics.json")
curves_path = Path("outputs/plots/pretrain_curves.png")
split_path = Path("outputs/processed/pretrain_split.json")
ckpt_path = Path("outputs/checkpoints/pretrain_best.pt")

for path in [metrics_path, curves_path, split_path, ckpt_path]:
    assert path.exists(), f"Missing {path}"

metrics = json.loads(metrics_path.read_text())
print("Pretrain metrics:", metrics)
display(Image(str(curves_path)))

split = json.loads(split_path.read_text())
print(
    f"pretrain_train={len(split['pretrain_train_ids'])}, "
    f"pretrain_val={len(split['pretrain_val_ids'])}"
)


## Fine-tune (load pretrained encoder)

Fine-tune on official train/val with the pretrained encoder weights. Checkpoint saved to `outputs/checkpoints/best_model.pt`.


In [ ]:
import yaml
from pathlib import Path

cfg_path = Path("configs/transformer.yaml")
cfg = yaml.safe_load(cfg_path.read_text())
cfg["wandb"]["run_name"] = "colab_mem_finetune_v1"
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(f"Set wandb run_name to {cfg['wandb']['run_name']}")


In [ ]:
!python scripts/train_transformer.py \
    --config configs/transformer.yaml \
    --pretrained-checkpoint outputs/checkpoints/pretrain_best.pt


## Test predictions

Generate submission probabilities and download `predictions.csv`.


In [ ]:
!python scripts/make_predictions.py \
    --checkpoint outputs/checkpoints/best_model.pt \
    --output outputs/predictions.csv


In [ ]:
from google.colab import files

files.download("outputs/predictions.csv")


## Optional next steps

- Upload to Hugging Face Hub:
  ```python
  # !python scripts/upload_to_hub.py --repo-id YOUR_USERNAME/ehr-timeline-transformer
  ```
- Compare against no-pretrain baseline: re-run fine-tune **without** `--pretrained-checkpoint` and compare `outputs/metrics/val_metrics.json`.
